# Spatial analysis-increment diagnostics

Process posterior-minus-prior increments for selected DA experiments without reading a reference experiment. This notebook is the spatial companion to `06_state_update_profiles.ipynb`. The primary combined map shows gridpoint RMS increments over the same 65-cycle common-S0 window used by notebook 06, with rows for PS and 1000-hPa U, V, T, and Q and columns for DART10-S0, DART20-S0, and DART40-S0. The vertical section remains a focused four-cycle case study for 26 December 2011.

## Notes: DA analysis increments

This workflow is independent of reference/observation comparisons. It reads only
the experiments listed in `DA_EXPERIMENTS` and computes:

| Metric | Meaning |
|---|---|
| `increment` | Posterior analysis mean minus prior forecast mean |
| `increment_rms` | Gridpoint RMS of the increment over the common-S0 window |

Available variables are `PS` at the surface and the 3-D variables `U`, `V`,
`T`, `Q`, `CLDLIQ`, and `CLDICE` at a selected pressure level. Add a `level`
for every 3-D request and omit it for `PS`. The lowest remapped pressure level
is 1000 hPa; it can be missing where that pressure surface lies below terrain.
Request-keyed cycle caches and common-window RMS caches are reused unless
`force_compute = True`.

For 3-D variables, `CROSS_SECTION_REQUESTS` derives time-mean vertical sections of signed or absolute analysis increments together with the posterior/prior spread ratio. A latitude-axis section uses a zonal mean; a longitude-axis section uses a cosine-latitude-weighted meridional mean. Separate request-keyed caches are reused unless `force_compute` is `True`.

## 1. Libraries


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


## 2. User setup: paths


In [2]:
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
SOURCE_WORKFLOW_NAME = "analysis_atm_init"  # supplies remapped ensemble files
WORKFLOW_NAME = "analysis_atm_da"  # owns this notebook's caches and figures


## 3. Project setup


In [3]:
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)
SOURCE_DATA_DIR, _ = workflow_output_dirs(
    SOURCE_WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
    create=False,
)

from util.atm_initialization_metrics import (
    DartAssimOutput,
    cache_map_metric,
    cache_increment_cross_section,
    increment_cross_section_cache_path,
    map_cache_path,
    plot_combined_cached_maps,
    plot_cached_increment_cross_sections,
)


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 4. User setup: diagnostic selections


In [4]:
ANALYSIS_DATA_DIR = SOURCE_DATA_DIR / "remapped_ensemble_diagnostics"
CACHE_DIR = WORKFLOW_DATA_DIR / "spatial_analysis_increments"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "spatial_analysis_increments"

DA_EXPERIMENTS = ["DART10-S0", "DART20-S0", "DART40-S0"]
# Map window matches notebook 06's common-S0 profile window exactly.
MAP_WINDOW_NAME = "common_s0"
MAP_WINDOW_LABEL = "16 December 2011–1 January 2012 (65 cycles)"
MAP_TIMESTAMPS = [
    f"{timestamp:%Y-%m-%d}-{timestamp.hour * 3600:05d}"
    for timestamp in pd.date_range(
        "2011-12-16 00:00:00", "2012-01-01 00:00:00", freq="6h"
    )
]

# The vertical section remains a representative mature-cycling-day case study.
CASE_STUDY_LABEL = "26 December 2011"
CROSS_SECTION_TIMESTAMPS = [
    "2011-12-26-00000",
    "2011-12-26-21600",
    "2011-12-26-43200",
    "2011-12-26-64800",
]

BOTTOM_LEVEL_HPA = 1000.0  # Lowest pressure level in the remapped files.
INCREMENT_REQUESTS = [
    {"variable": "PS", "label": "PS", "scale": 0.01, "unit": "hPa"},
    {"variable": "U", "level": BOTTOM_LEVEL_HPA, "label": "U at 1000 hPa", "scale": 1.0, "unit": "m s$^{-1}$"},
    {"variable": "V", "level": BOTTOM_LEVEL_HPA, "label": "V at 1000 hPa", "scale": 1.0, "unit": "m s$^{-1}$"},
    {"variable": "T", "level": BOTTOM_LEVEL_HPA, "label": "T at 1000 hPa", "scale": 1.0, "unit": "K"},
    {"variable": "Q", "level": BOTTOM_LEVEL_HPA, "label": "Q at 1000 hPa", "scale": 1000.0, "unit": "g kg$^{-1}$"},
]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
force_compute = False
RUN_WORKFLOW = True
COMBINED_MAP_FIGSIZE = (16, 15)
COMBINED_MAP_TITLE_TEMPLATE = None
# Example: COMBINED_MAP_TITLE_TEMPLATE = "RMS analysis increments | {window}"
fontz = 18
SHOW_COMBINED_MAP = True
FIGURE_FORMAT = "pdf"
if len(MAP_TIMESTAMPS) != 65:
    raise ValueError("The common-S0 map window must contain 65 cycles.")
if FIGURE_FORMAT not in {"pdf", "png"}:
    raise ValueError("FIGURE_FORMAT must be either 'pdf' or 'png'.")

SURFACE_VARIABLES = {"PS"}
THREE_D_VARIABLES = {"U", "V", "T", "Q", "CLDLIQ", "CLDICE"}

MODEL_LABELS = {
    "CAM6-S0": "CAM6-DART (ENS=80)",
    "CTRL10-S0": "CTRL (ENS=10)",
    "DART10-S0": "DART10-S0",
    "DART20-S0": "DART20-S0",
    "DART40-S0": "DART40-S0",
    "DART40INF0p6-S0": "EAM-DART (ENS=40, INF=0.6)",
}

# Vertical DA-increment sections; each request is cached for every timestamp.
CROSS_SECTION_REQUESTS = [
    {"variable": "T", "horizontal_axis": "latitude", "increment_mode": "absolute"},
]

FIGURE_DPI = 300


## 5. Read, process, and cache


In [5]:
if RUN_WORKFLOW:
    for request in INCREMENT_REQUESTS:
        variable = request["variable"]
        level = request.get("level")
        if variable not in SURFACE_VARIABLES | THREE_D_VARIABLES:
            raise ValueError(f"Unsupported variable: {variable}")
        if variable in THREE_D_VARIABLES and level is None:
            raise ValueError(f"A level is required for 3-D variable {variable}")
        if variable in SURFACE_VARIABLES and level is not None:
            raise ValueError(f"Surface variable {variable} must not specify a level")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in MAP_TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request.get("level")):
                map_cache_path(
                    CACHE_DIR, experiment, timestamp, request["variable"],
                    "increment", request.get("level")
                )
            for experiment in DA_EXPERIMENTS
            for request in INCREMENT_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all increments exist; reusing cache.")
            continue

        print(f"{timestamp}: deriving missing/forced DA increments.")
        outputs = {
            experiment: DartAssimOutput.from_experiment(
                ANALYSIS_DATA_DIR, experiment, timestamp
            )
            for experiment in DA_EXPERIMENTS
        }
        try:
            for (experiment, variable, level), path in paths.items():
                dataset = cache_map_metric(
                    path, outputs[experiment], None, variable, "increment",
                    LAT_RANGE, LON_RANGE, level=level,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (
                    output.prior_mean, output.prior_sd,
                    output.post_mean, output.post_sd,
                ):
                    dataset.close()


if RUN_WORKFLOW and CROSS_SECTION_REQUESTS:
    for request in CROSS_SECTION_REQUESTS:
        variable = request["variable"]
        horizontal_axis = request.get("horizontal_axis", "latitude")
        increment_mode = request.get("increment_mode", "absolute")
        if variable not in THREE_D_VARIABLES:
            raise ValueError(f"Cross-section variable must be 3-D: {variable}")
        if horizontal_axis not in {"latitude", "longitude"}:
            raise ValueError(f"Unsupported horizontal_axis: {horizontal_axis}")
        if increment_mode not in {"signed", "absolute"}:
            raise ValueError(f"Unsupported increment_mode: {increment_mode}")

    for timestamp in CROSS_SECTION_TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request.get("horizontal_axis", "latitude"), request.get("increment_mode", "absolute")): increment_cross_section_cache_path(
                CACHE_DIR, experiment, timestamp, request["variable"],
                request.get("horizontal_axis", "latitude"),
                request.get("increment_mode", "absolute"),
            )
            for experiment in DA_EXPERIMENTS
            for request in CROSS_SECTION_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all increment sections exist; reusing cache.")
            continue
        outputs = {experiment: DartAssimOutput.from_experiment(ANALYSIS_DATA_DIR, experiment, timestamp) for experiment in DA_EXPERIMENTS}
        try:
            for (experiment, variable, horizontal_axis, increment_mode), path in paths.items():
                dataset = cache_increment_cross_section(
                    path, outputs[experiment], variable, LAT_RANGE, LON_RANGE,
                    horizontal_axis=horizontal_axis,
                    increment_mode=increment_mode,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (output.prior_mean, output.prior_sd, output.post_mean, output.post_sd):
                    dataset.close()


2011-12-16-00000: deriving missing/forced DA increments.


FileNotFoundError: /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_init/remapped_ensemble_diagnostics/DART10-S0/DART10-S0.forecast_mean.2011-12-16-00000.nc

## 6. Plot, save, and show


In [ ]:
def window_rms_map_cache_path(experiment, request):
    variable = request["variable"]
    level = request.get("level")
    level_tag = "surface" if level is None else f"level-{level:g}"
    return CACHE_DIR / (
        f"{experiment}_{MAP_WINDOW_NAME}_map_{variable}_"
        f"increment-rms_{level_tag}.nc"
    )


def read_or_aggregate_window_rms_map(experiment, request):
    target = window_rms_map_cache_path(experiment, request)
    if target.is_file() and not force_compute:
        cached = xr.load_dataset(target)
        if cached.attrs.get("number_of_cycles") == len(MAP_TIMESTAMPS):
            print("[REUSE]", target)
            return cached
        cached.close()

    variable = request["variable"]
    level = request.get("level")
    paths = [
        map_cache_path(
            CACHE_DIR, experiment, timestamp, variable, "increment", level
        )
        for timestamp in MAP_TIMESTAMPS
    ]
    missing = [path for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing increment caches:\n" + "\n".join(str(path) for path in missing)
        )
    snapshots = [xr.load_dataset(path) for path in paths]
    try:
        increments = xr.concat(
            [dataset["metric"] for dataset in snapshots],
            dim=xr.IndexVariable("analysis_time", MAP_TIMESTAMPS),
        )
        result = np.sqrt((increments ** 2).mean("analysis_time", skipna=True))
        result = result.to_dataset(name="metric").load()
        result.attrs.update({
            "experiment": experiment,
            "variable": variable,
            "level": "surface" if level is None else float(level),
            "metric": "increment_rms",
            "window": MAP_WINDOW_NAME,
            "window_label": MAP_WINDOW_LABEL,
            "number_of_cycles": len(MAP_TIMESTAMPS),
        })
        result.to_netcdf(target, mode="w")
        print("[WRITE]", target)
        return result
    finally:
        for dataset in snapshots:
            dataset.close()


if RUN_WORKFLOW:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    combined_rms_maps = {}
    for request in INCREMENT_REQUESTS:
        request_key = (request["variable"], request.get("level"))
        combined_rms_maps[request_key] = {
            experiment: read_or_aggregate_window_rms_map(experiment, request)
            for experiment in DA_EXPERIMENTS
        }
    try:
        combined_title = (
            None if not COMBINED_MAP_TITLE_TEMPLATE
            else COMBINED_MAP_TITLE_TEMPLATE.format(window=MAP_WINDOW_LABEL)
        )
        figure = plot_combined_cached_maps(
            combined_rms_maps, INCREMENT_REQUESTS, DA_EXPERIMENTS,
            LAT_RANGE, LON_RANGE, label_map=MODEL_LABELS,
            title=combined_title, figsize=COMBINED_MAP_FIGSIZE,
            font_size=fontz,
        )
        combined_output = FIGURE_DIR / (
            f"state_update_rms_maps_{MAP_WINDOW_NAME}.{FIGURE_FORMAT}"
        )
        figure.savefig(combined_output, dpi=FIGURE_DPI, bbox_inches="tight")
        print("[FIGURE]", combined_output)
        if SHOW_COMBINED_MAP:
            plt.show()
        else:
            plt.close(figure)
    finally:
        for experiment_maps in combined_rms_maps.values():
            for dataset in experiment_maps.values():
                dataset.close()


if RUN_WORKFLOW and CROSS_SECTION_REQUESTS:
    for request in CROSS_SECTION_REQUESTS:
        variable = request["variable"]
        horizontal_axis = request.get("horizontal_axis", "latitude")
        increment_mode = request.get("increment_mode", "absolute")
        time_means = {}
        for experiment in DA_EXPERIMENTS:
            paths = [
                increment_cross_section_cache_path(
                    CACHE_DIR, experiment, timestamp, variable,
                    horizontal_axis, increment_mode,
                )
                for timestamp in CROSS_SECTION_TIMESTAMPS
            ]
            missing = [path for path in paths if not path.is_file()]
            if missing:
                raise FileNotFoundError("Missing increment-section caches:\n" + "\n".join(str(path) for path in missing))
            snapshots = [xr.load_dataset(path) for path in paths]
            try:
                time_mean = xr.concat(
                    snapshots, dim=xr.IndexVariable("analysis_time", CROSS_SECTION_TIMESTAMPS)
                ).mean("analysis_time", keep_attrs=True)
                time_mean.attrs["analysis_time"] = "time mean: " + ", ".join(CROSS_SECTION_TIMESTAMPS)
                time_means[experiment] = time_mean.load()
            finally:
                for dataset in snapshots:
                    dataset.close()
        try:
            figure = plot_cached_increment_cross_sections(
                time_means, variable, increment_mode=increment_mode,
                label_map=MODEL_LABELS,
                title=(
                    f"{variable} {increment_mode.title()} Analysis Increment "
                    f"and Posterior/Prior Spread Ratio | {CASE_STUDY_LABEL}"
                ),
            )
            figure.savefig(
                FIGURE_DIR / f"{variable}_{increment_mode}_increment_{horizontal_axis}_time-mean.{FIGURE_FORMAT}",
                dpi=FIGURE_DPI, bbox_inches="tight",
            )
            plt.close(figure)
        finally:
            for dataset in time_means.values():
                dataset.close()
